In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, desc, count, when, col, round, floor, sum
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Flight Delay EDA") \
    .getOrCreate()

# Read Sample data
sample = spark.read.csv("../data/sample.csv", header=True, inferSchema=True)
sample.show(5)

+-----+------------+-----------+--------------------+-----------------+------+--------------------+----+--------------+------------+---------+------------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|ORIGIN|    ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|CRS_DEP_TIME|DEP_DELAY|CRS_ARR_TIME|ARR_DELAY|CANCELLED|DIVERTED|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+-----+------------+-----------+--------------------+-----------------+------+--------------------+----+--------------+------------+---------+------------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|    1|           1|          7|1/1/2023 12:00:00 AM|               9E|   ABE|Allentown/Bethleh...| ATL|   Atlanta, GA|         600|      1.0|         825|     -9.0|      0.0|     0.0|   692.0|      

# Which airports have the highest delay rate on average? 

In [51]:
# Count total flights and delays for each airport
# Delayed flights are those with an arrival delay of 15 minutes or more
airport_delays = sample.groupBy("ORIGIN", "ORIGIN_CITY_NAME").agg(
    count("*").alias("total_flights"),
    count(when(col("ARR_DELAY") >= 15, True)).alias("delayed_flights")
)

# Calculate delay rate
airport_delays = airport_delays.withColumn(
    "delay_rate",
    round(col("delayed_flights") / col("total_flights") * 100, 2).alias("delay_rate")
)

# Order by delay rate in descending order
airport_delays = airport_delays.orderBy(desc("delay_rate"))

# Show which airports have the highest delay rates
airport_delays.show()

+------+--------------------+-------------+---------------+----------+
|ORIGIN|    ORIGIN_CITY_NAME|total_flights|delayed_flights|delay_rate|
+------+--------------------+-------------+---------------+----------+
|   CSG|        Columbus, GA|            2|              2|     100.0|
|   ECP|     Panama City, FL|            1|              1|     100.0|
|   AEX|      Alexandria, LA|            2|              2|     100.0|
|   EUG|          Eugene, OR|            2|              2|     100.0|
|   BQK|       Brunswick, GA|            2|              2|     100.0|
|   CRW|Charleston/Dunbar...|            2|              2|     100.0|
|   JAC|         Jackson, WY|            3|              3|     100.0|
|   BTR|     Baton Rouge, LA|            2|              2|     100.0|
|   SLC|  Salt Lake City, UT|            9|              8|     88.89|
|   BOI|           Boise, ID|            4|              3|      75.0|
|   PDX|        Portland, OR|            6|              4|     66.67|
|   FA

# Which airlines have the highest delay rates on average?

In [53]:
# Count total flights and delays for each airline
# Delayed flights are those with an arrival delay of 15 minutes or more
airline_delays = sample.groupBy("OP_UNIQUE_CARRIER").agg(
    count("*").alias("total_flights"),
    count(when(col("ARR_DELAY") >= 15, True)).alias("delayed_flights")
)

# Calculate delay rate
airline_delays = airline_delays.withColumn(
    "delay_rate",
    round(col("delayed_flights") / col("total_flights") * 100, 2).alias("delay_rate")
)

# Order by delay rate in descending order
airline_delays = airline_delays.orderBy(desc("delay_rate"))

# Show which airlines have the highest delay rates
airline_delays.show()

+-----------------+-------------+---------------+----------+
|OP_UNIQUE_CARRIER|total_flights|delayed_flights|delay_rate|
+-----------------+-------------+---------------+----------+
|               AA|         2150|            413|     19.21|
|               9E|          350|             44|     12.57|
+-----------------+-------------+---------------+----------+



# What times of day do airports have the worst delays?

In [47]:
# Show the average arrival delay for each departure hour
time_delay = sample.withColumn(
    "departure_hour",
    floor(col("CRS_DEP_TIME") / 100).alias("departure_hour")
)

hourly_delay_rate = (time_delay.withColumn(
        "is_delayed",
        when(col("ARR_DELAY") >= 15, 1).otherwise(0)
    ).groupBy("departure_hour").agg(
        count("*").alias("total_flights"),
        round(avg("is_delayed"), 2).alias("delay_rate")
    )
    .orderBy("departure_hour")
)

hourly_delay_rate.show()


+--------------+-------------+----------+
|departure_hour|total_flights|delay_rate|
+--------------+-------------+----------+
|             0|            1|       0.0|
|             1|            1|       0.0|
|             5|           26|      0.15|
|             6|          121|      0.13|
|             7|          127|      0.16|
|             8|          124|      0.14|
|             9|          126|      0.13|
|            10|          167|      0.16|
|            11|          154|      0.23|
|            12|          182|      0.19|
|            13|          174|       0.2|
|            14|          160|      0.16|
|            15|          140|      0.15|
|            16|          165|       0.2|
|            17|          151|      0.21|
|            18|          181|      0.21|
|            19|          147|      0.18|
|            20|          132|       0.2|
|            21|          102|      0.25|
|            22|           96|      0.22|
+--------------+-------------+----

# What delay type is most common?

In [ ]:
# Create flags for each type of delay
df_flags = sample.withColumn(
    "carrier",
    when(col("CARRIER_DELAY") > 0, 1).otherwise(0)
).withColumn(
    "weather",
    when(col("WEATHER_DELAY") > 0, 1).otherwise(0)
).withColumn(
    "nas",
    when(col("NAS_DELAY") > 0, 1).otherwise(0)
).withColumn(
    "security",
    when(col("SECURITY_DELAY") > 0, 1).otherwise(0)
).withColumn(
    "late_aircraft",
    when(col("LATE_AIRCRAFT_DELAY") > 0, 1).otherwise(0)
)

# Show the average delay rate for each type of delay
df_flags.agg(
    round(avg("carrier"), 4).alias("carrier_rate"),
    round(avg("weather"), 4).alias("weather_rate"),
    round(avg("nas"), 4).alias("nas_rate"),
    round(avg("security"), 4).alias("security_rate"),
    round(avg("late_aircraft"), 4).alias("late_aircraft_rate")
).show()

+------------+------------+--------+-------------+------------------+
|carrier_rate|weather_rate|nas_rate|security_rate|late_aircraft_rate|
+------------+------------+--------+-------------+------------------+
|      0.1084|      0.0084|  0.0936|       0.0016|            0.0772|
+------------+------------+--------+-------------+------------------+

